# Day 2.9 — Pivotal Exercise: Assemble RAG Context
This is the day's one hands-on implementation lab. It uses no API key. Try the starter cell first; a fully commented reference solution follows the check so you can compare or catch up.


## Why this mechanism matters

Retrieval results are not automatically model context. The application must select, order, label, and limit evidence. That boundary affects grounding, citations, latency, and resistance to irrelevant text.

## Contract

Implement `build_context`. Preserve rank order, label each included chunk as `[source | section]`, stay within the character budget, and skip rather than truncate a chunk that does not fit.

Before coding, write one sentence predicting the easiest mistake to make.

In [ ]:
def build_context(chunks, character_budget):
    """Return one string of labelled evidence blocks that fits the budget.

    Each chunk is {"source": str, "section": str, "text": str}.
    Block format:  [source | section]\ntext      Blocks are separated by a blank line.
    """
    # TODO: walk the chunks in the given (ranked) order
    # TODO: build the labelled block for each chunk
    # TODO: include the block only if the whole block still fits the budget
    # TODO: join the included blocks with "\n\n"
    raise NotImplementedError("Complete context assembly")

## Behavioural check

Run this after completing the starter cell. If you have not finished, it prints a hint instead of failing. A passing check proves the listed contract examples, not every possible input.

In [ ]:
def run_checks():
    chunks = [
        {"source": "a.md", "section": "Safety", "text": "Wear eye protection."},
        {"source": "b.md", "section": "Power", "text": "Verify protective earth."},
        {"source": "c.md", "section": "Noise", "text": "This distractor should not fit."},
    ]
    result = build_context(chunks, 90)
    print("Assembled context (", len(result), "characters ):")
    print(result)
    assert len(result) <= 90, "the budget is a hard limit"
    assert "[a.md | Safety]" in result and "Wear eye protection." in result
    assert "[b.md | Power]" in result and "Verify protective earth." in result
    assert result.index("[a.md") < result.index("[b.md"), "rank order must be preserved"
    assert "This distractor" not in result, "a chunk that does not fit is skipped, never cut"
    assert "[c.md" not in result, "a skipped chunk must not leave a dangling label"
    print("PASS: context is labelled, ordered, bounded, and never truncated")

try:
    run_checks()
except NotImplementedError:
    print("Not implemented yet. Complete the starter cell above, or study the reference solution below and re-run this cell.")

## Reference solution

Read this even if your check passed: compare each commented line with your version, then re-run the check cell above.

In [ ]:
# --- Reference solution: read it line by line, then re-run the check cell above ---
def build_context(chunks, character_budget):
    blocks = []                                   # the blocks we decided to include
    used = 0                                      # characters spent so far
    for chunk in chunks:                          # ranked order in, ranked order out
        block = f"[{chunk['source']} | {chunk['section']}]\n{chunk['text']}"
        separator = 2 if blocks else 0            # "\n\n" costs 2 characters between blocks
        if used + separator + len(block) > character_budget:
            print(f"skip  {chunk['source']}: {len(block)} chars would exceed the budget")
            continue                              # skip the WHOLE chunk; never cut it in half
        blocks.append(block)
        used += separator + len(block)
        print(f"keep  {chunk['source']}: {used}/{character_budget} chars used")
    return "\n\n".join(blocks)

print("Reference build_context defined. Re-run the check cell above to see PASS.")

## Explain

**What changes when top-k grows but the context budget does not?**

<details><summary>Show answer</summary>

More candidates compete for the same space. Lower-ranked chunks are skipped, so a larger top-k only helps if the ranking is good; if it is poor, a relevant chunk ranked 6th is still lost. This is why Day 2.6 measures retrieval separately.

</details>

**Why is skipping a chunk better than truncating it?**

<details><summary>Show answer</summary>

A half chunk can end mid-sentence and change meaning, and its citation label would point at text the model never saw. Complete blocks keep every citation verifiable.

</details>